# Silver Analysis Cookbook — 5 recipes

On `data/silver/compliance_chunks.parquet` (1519 chunks). No embeddings, no GPU. Copy-paste each recipe.

**Recipes:** 1 filter → 2 keyword/TF-IDF search → 3 cross-framework compare → 4 domain rollup → 5 export for RAG

> Cite `framework_id` + `source` + `sha256` for every answer.

In [1]:
import pandas as pd
from pathlib import Path

parquet = Path("../data/silver/compliance_chunks.parquet")
if not parquet.exists():
    parquet = Path("data/silver/compliance_chunks.parquet")
if not parquet.exists():
    from datasets import load_dataset
    df = load_dataset("GauravGurjar/open-regulatory-corpus")["train"].to_pandas()
else:
    df = pd.read_parquet(parquet)
print(f"{len(df)} chunks, {df.framework_id.nunique()} framework_ids")


1519 chunks, 54 framework_ids


## 1 — Filter by framework / domain
Fastest path to a corpus slice. Use `framework_id` stem; `_pdf` suffix = secondary PDF for same framework.

In [2]:
# single framework
fda = df[df.framework_id == "FDA-AI-MD"]
print(f"FDA-AI-MD: {len(fda)} chunks")
print(fda.iloc[0].text[:600].replace("\n"," ")[:600])
print("source:", fda.iloc[0].source, "|", fda.iloc[0].sha256[:12])

# all PDFs for a framework (e.g. SOX has _pdf1/_pdf2)
sox = df[df.framework_id.str.startswith("SOX")]
print(f"\nSOX* : {len(sox)} chunks across", sox.framework_id.unique().tolist())

# domain slice — privacy + AI governance (see 02 for domain map)
privacy = ["CCPA-CPRA","FERPA","COPPA","AU-PRIVACY-AI","UK-DP-AI","DOJ-DSP"]
ai_gov = ["NIST-AI-RMF","NIST-AI-600-1","OMB-M25-21","OWASP-LLM-2026","SG-MODEL-AI"]
subset = df[df.framework_id.str.split("_pdf").str[0].isin(privacy + ai_gov)]
print(f"\nprivacy+ai: {len(subset)} chunks")
print(subset.framework_id.value_counts().head(6).to_string())


FDA-AI-MD: 80 chunks
Artificial Intelligence-Enabled Medical Devices The FDA encourages the development of innovative, safe, and effective medical devices, including devices that incorporate artificial intelligence (AI). The AI-Enabled Medical Device List is a resource intended to identify AI-enabled medical devices that are authorized for marketing in the United States. Digital health innovators can refer to this list to gain insights into the current device landscape and regulatory expectations, which can help foster innovation and ensure public safety. This list can also provide transparency for healthcare prov
source: data/bronze/latest/raw/FDA-AI-MD.html | c8a961fb9fdc

SOX* : 226 chunks across ['SOX', 'SOX_pdf1', 'SOX_pdf2']

privacy+ai: 273 chunks
framework_id
CCPA-CPRA_pdf2    121
FERPA              39
AU-PRIVACY-AI      38
OMB-M25-21         29
SG-MODEL-AI        19
UK-DP-AI            7


## 2 — Keyword search (TF-IDF, no embeddings)
`sklearn` if present, else pure `str.contains`. Both cite source.

In [3]:
query = "incident response logging"

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    vec = TfidfVectorizer(stop_words="english", max_features=5000)
    X = vec.fit_transform(df.text.tolist())
    q = vec.transform([query])
    scores = cosine_similarity(q, X).flatten()
    top = scores.argsort()[::-1][:5]
    for i in top:
        r = df.iloc[i]
        print(f"{scores[i]:.3f}  {r.framework_id:18} {r.chunk_id}")
        print(r.text[:380].replace("\n"," ")[:380])
        print(f"→ {r.source}  sha:{r.sha256[:8]}\n")
except ImportError:
    hits = df[df.text.str.contains(query, case=False, na=False)].head(5)
    for _, r in hits.iterrows():
        print(r.framework_id, r.chunk_id)
        print(r.text[:380].replace("\n"," ")[:380])
        print(f"→ {r.source}\n")


0.503  IRS-1075_pdf1      IRS-1075_pdf1-137
mechanisms can provide a push and/or pull capability for users to obtain incident response assistance. For example, individuals might have access to a website to query the assistance capability, or the assistance capability can proactively send information to users (general distribution or targeted) as part of increasing understanding of current response capabilities and suppor
→ data/bronze/latest/raw/IRS-1075_pdf1.pdf  sha:013e5255

0.500  IRS-1075_pdf1      IRS-1075_pdf1-134
Addresses purpose, scope, roles, responsibilities, management commitment, coordination among organizational entities and compliance; and (b) Is consistent with applicable laws, Executive Orders, directives, regulations, policies, standards, and guidelines; and 2. Procedures to facilitate the implementation of the incident response policy and the associated Incident response con
→ data/bronze/latest/raw/IRS-1075_pdf1.pdf  sha:013e5255

0.495  CJIS-6.1_pdf       CJIS-6.1

## 3 — Cross-framework comparison
Same query, ranked per-framework. Useful for: does NIST-CSF2 say X but SOX says Y?

In [4]:
# compare two frameworks on one theme
query = "risk management"
frameworks = ["NIST-CSF2", "FDA-AI-MD", "SOX", "IRS-1075"]
# normalize stems so _pdf variants count
df["stem"] = df.framework_id.str.split("_pdf").str[0]

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    vec = TfidfVectorizer(stop_words="english", max_features=5000)
    X = vec.fit_transform(df.text.tolist())
    q = vec.transform([query])
    df["score"] = cosine_similarity(q, X).flatten()
    for fid in frameworks:
        sub = df[df.stem == fid].sort_values("score", ascending=False).head(2)
        print(f"\n=== {fid} ({len(df[df.stem==fid])} chunks) ===")
        for _, r in sub.iterrows():
            print(f"  {r.score:.3f} {r.chunk_id}: {r.text[:220].replace(chr(10),' ')[:220]}...")
except ImportError:
    for fid in frameworks:
        hits = df[(df.stem==fid) & df.text.str.contains(query, case=False, na=False)].head(2)
        print(f"\n=== {fid} ===")
        for _, r in hits.iterrows():
            print(r.chunk_id, r.text[:220].replace(chr(10)," ")[:220])



=== NIST-CSF2 (28 chunks) ===
  0.488 NIST-CSF2-14: to balance a portfolio of risk considerations, including cybersecurity, and make informed decisions. Executives receive significant input about current and planned risk activities as they integrate governance and risk st...
  0.392 NIST-CSF2-13: needs, they communicate expectations about risk appetite, accountability, and resources. Executives are also responsible for integrating cybersecurity risk management with ERM programs and lower-level risk management pro...

=== FDA-AI-MD (80 chunks) ===
  0.011 FDA-AI-MD-68: Imaging System | Canon Medical Systems Corporation | Radiology | JAK | | 02/12/2020 | K193178 | Biograph Horizon | Siemens Medical Solution USA, Inc. | Radiology | KPS | | 02/10/2020 | K192377 | Ethos Treatment Managemen...
  0.011 FDA-AI-MD-29: K241596 | uMI Panvivo | Shanghai United Imaging Healthcare Co., Ltd. | Radiology | KPS | | 08/15/2024 | K241211 | CoLumbo | Smart Soft Healthcare | Radiology | QIH | | 08/13/2024

## 4 — Domain rollup & simple overlap
How much vocabulary do two domains share? Cheap Jaccard on top terms — no LLM needed.

In [5]:
import re
from collections import Counter

STOP = set("the and for are with this that from have will shall must".split())
def top_terms(texts, k=30):
    c = Counter()
    for t in texts:
        toks = re.findall(r"[a-z]{3,}", t.lower())
        c.update(w for w in toks if w not in STOP)
    return set(w for w,_ in c.most_common(k))

a_terms = top_terms(df[df.stem=="NIST-CSF2"].text.tolist())
b_terms = top_terms(df[df.stem=="SOX"].text.tolist())
jaccard = len(a_terms & b_terms) / len(a_terms | b_terms) if a_terms|b_terms else 0
print(f"NIST-CSF2 ∩ SOX  jaccard@{len(a_terms)} = {jaccard:.2f}")
print("shared:", sorted(a_terms & b_terms)[:15])
print("only CSF2:", sorted(a_terms - b_terms)[:10])
print("only SOX:", sorted(b_terms - a_terms)[:10])

# quick domain size table
print("\n", df.groupby("stem").size().sort_values(ascending=False).head(8).to_string())


NIST-CSF2 ∩ SOX  jaccard@30 = 0.09
shared: ['information', 'its', 'other', 'page', 'their']
only CSF2: ['activities', 'can', 'core', 'csf', 'cswp', 'cybersecurity', 'february', 'framework', 'incident', 'management']
only SOX: ['accounting', 'agencies', 'assets', 'balance', 'commission', 'credit', 'financial', 'has', 'issuer', 'issuers']

 stem
CJIS-6.1         466
IRS-1075         228
SOX              226
CCPA-CPRA        126
EAR               84
FDA-AI-MD         80
AU-PRIVACY-AI     42
FERPA             39


## 5 — Export a RAG-ready slice
Write a filtered parquet/jsonl with provenance — drop into any Gold/embed pipeline.

In [6]:
from pathlib import Path
import json

# example: export all Financial + Cybersecurity chunks
keep_stems = {"SOX","GLBA","NYDFS-500","FRB-MRM-2026","ECOA-REG-B","NAIC-AI",
              "NIST-CSF2","NIST-800-53","NIST-800-171","FEDRAMP","CJIS-6.1","IRS-1075"}
slice_df = df[df.stem.isin(keep_stems)][["framework_id","chunk_id","text","source","sha256","token_est"]]
print(f"slice: {len(slice_df)} chunks")

# preview jsonl (one chunk per line — standard for embedding jobs)
preview = slice_df.head(2).to_dict(orient="records")
for r in preview:
    print(json.dumps({k: (v[:120]+"..." if k=="text" else v) for k,v in r.items()}, indent=2))

# uncomment to write:
# out = Path("../data/silver/fin_cyber_slice.parquet")
# slice_df.to_parquet(out, index=False)
# print(f"wrote {out} ({out.stat().st_size/1e6:.1f} MB)")
# slice_df.to_json("fin_cyber_slice.jsonl", orient="records", lines=True)


slice: 1010 chunks
{
  "framework_id": "CJIS-6.1",
  "chunk_id": "CJIS-6.1-0",
  "text": "CJIS Security Policy v6.1 2026-06-25 &#8212; LE {\"@context\": \"http://schema.org\", \"headline\": \"CJIS Security Policy v6.1...",
  "source": "data/bronze/latest/raw/CJIS-6.1.html",
  "sha256": "dadc3fdb12afc47195beedc0f8a8b35f52d0b90fc39a4b2a0efc8c7ca2c1c44a",
  "token_est": 510.72
}
{
  "framework_id": "CJIS-6.1",
  "chunk_id": "CJIS-6.1-1",
  "text": "fbi federal bureau of investigation FBI.gov Contact Center Email updates Accessibility eRulemaking Freedom of Informatio...",
  "source": "data/bronze/latest/raw/CJIS-6.1.html",
  "sha256": "dadc3fdb12afc47195beedc0f8a8b35f52d0b90fc39a4b2a0efc8c7ca2c1c44a",
  "token_est": 196.84
}


## Next
- Embed this slice with `sentence-transformers/all-MiniLM-L6-v2` + FAISS (Gold phase).
- Track provenance: every answer cites `framework_id` + `source` + `sha256` → `data/sources.json`.
- See `02_silver_eda.ipynb` for what's *not* in Silver yet (WAF gaps).